# Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import json
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from sklearn.model_selection import train_test_split


In [4]:
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType
from torch.utils.data import Dataset

In [5]:
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU: Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# CONFIGURATION

In [20]:
@dataclass
class Config:
    """Centralized configuration"""
    # Paths
    data_dir: str = "/content/drive/MyDrive/DataNLP/"
    output_dir: str = "/content/drive/MyDrive/outputs"
    adapter_dir: str = "/content/drive/MyDrive/adapters"

    # Model
    backbone_model: str = "microsoft/deberta-v3-small"
    max_length: int = 512

    # LoRA Configuration
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.2

    # Training
    num_epochs: int = 5
    batch_size: int = 8
    eval_batch_size: int = 32
    gradient_accumulation_steps: int = 2
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.15
    weight_decay: float = 0.05

    # Early stopping
    early_stopping_patience: int = 5
    early_stopping_threshold: float = 0.001

    # Domains
    domains: List[str] = None

    def __post_init__(self):
        if self.domains is None:
            self.domains = [
                "phishing", "fake_news", "political_statements",
                "product_reviews", "job_scams", "sms", "twitter_rumours"
            ]
        # Create directories
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        Path(self.adapter_dir).mkdir(parents=True, exist_ok=True)

config = Config()

# DATA LOADING

In [8]:
class DataManager:
    """Handles all data loading and preprocessing"""

    def __init__(self, config: Config):
        self.config = config
        self.domain_data = {}

    def load_all_domains(self) -> Dict[str, pd.DataFrame]:
        """Load all domain datasets"""
        print("\n📂 Loading domain datasets...")

        for domain in self.config.domains:
            file_path = f"/content/drive/MyDrive/DataNLP/processed_{domain}.pkl"
            try:
                with open(file_path, "rb") as f:
                    self.domain_data[domain] = pickle.load(f)
                print(f"  ✅ {domain}: {len(self.domain_data[domain])} samples")
            except Exception as e:
                print(f"  ❌ {domain}: Failed to load - {e}")

        print(f"\n✅ Loaded {len(self.domain_data)} domains successfully")
        return self.domain_data

    def get_domain_stats(self) -> pd.DataFrame:
        """Get statistics for all domains"""
        stats = []
        for domain, df in self.domain_data.items():
            class_counts = df["label"].value_counts().sort_index()
            stats.append({
                "domain": domain,
                "total": len(df),
                "legitimate": class_counts.get(0, 0),
                "fraud": class_counts.get(1, 0),
                "fraud_ratio": class_counts.get(1, 0) / len(df)
            })
        return pd.DataFrame(stats)

    def create_unified_dataset(self) -> pd.DataFrame:
        """Create unified dataset with domain labels for domain classifier"""
        unified_data = []

        for domain_idx, (domain, df) in enumerate(self.domain_data.items()):
            df_copy = df.copy()
            df_copy["domain"] = domain
            df_copy["domain_id"] = domain_idx
            unified_data.append(df_copy)

        return pd.concat(unified_data, ignore_index=True)


# DATASET CLASSES

In [9]:
class FraudDataset(Dataset):
    """Dataset for fraud detection task"""
    def __init__(self, df: pd.DataFrame):
        self.input_ids = torch.tensor(df["input_ids"].tolist(), dtype=torch.long)
        self.attention_mask = torch.tensor(df["attention_mask"].tolist(), dtype=torch.long)
        self.labels = torch.tensor(df["label"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


In [10]:
class MultiTaskDataset(Dataset):
    """Dataset for multi-task learning (fraud + domain classification)"""
    def __init__(self, df: pd.DataFrame):
        self.input_ids = torch.tensor(df["input_ids"].tolist(), dtype=torch.long)
        self.attention_mask = torch.tensor(df["attention_mask"].tolist(), dtype=torch.long)
        self.fraud_labels = torch.tensor(df["label"].tolist(), dtype=torch.long)
        self.domain_labels = torch.tensor(df["domain_id"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.fraud_labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "fraud_labels": self.fraud_labels[idx],
            "domain_labels": self.domain_labels[idx],
        }


In [11]:
class LazyMultiTaskDataset(Dataset):
    """
    Lazy-loading dataset for multi-task learning (fraud + domain classification).
    Optimized to avoid repeated pickle loading.
    """

    def __init__(self, domain_files, tokenizer, max_length=512, val_ratio=0.1, is_val=False):
        self.domain_files = domain_files
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.index_map = []          # (file_path, row_index, domain_id)
        self._df_cache = {}          # per-worker dataframe cache

        for domain_id, file_path in enumerate(domain_files):
            with open(file_path, "rb") as f:
                df = pickle.load(f)

            # Optional validation split
            if val_ratio > 0:
                n_val = int(len(df) * val_ratio)
                val_idx = df.sample(n=n_val, random_state=42).index
                if is_val:
                    df = df.loc[val_idx]
                else:
                    df = df.drop(val_idx)

            self.index_map.extend(
                [(file_path, i, domain_id) for i in range(len(df))]
            )

    def __len__(self):
        return len(self.index_map)

    def _get_df(self, file_path):
        """
        Load dataframe once per worker and cache it.
        """
        if file_path not in self._df_cache:
            with open(file_path, "rb") as f:
                self._df_cache[file_path] = pickle.load(f)
        return self._df_cache[file_path]

    def __getitem__(self, idx):
        file_path, row_idx, domain_id = self.index_map[idx]

        df = self._get_df(file_path)
        row = df.iloc[row_idx]

        encoding = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "fraud_labels": torch.tensor(row["label"], dtype=torch.long),
            "domain_labels": torch.tensor(domain_id, dtype=torch.long),
        }


# ENHANCED MODEL WITH MULTI-TASK LEARNING

In [12]:
class MultiTaskFraudDetector(nn.Module):
    """
    Multi-task model with:
    1. Shared LoRA adapter (cross-domain patterns)
    2. Domain classifier head
    3. Fraud classifier head
    """

    def __init__(self, backbone, num_domains: int):
        super().__init__()
        self.backbone = backbone
        self.num_domains = num_domains

        # Get hidden size from backbone
        hidden_size = backbone.config.hidden_size

        # Domain classifier head
        self.domain_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, num_domains)
        )

        # Fraud classifier head (replacing backbone's original classifier)
        self.fraud_classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 2)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        fraud_labels=None,
        domain_labels=None,
        return_embeddings=False
    ):
        # Get backbone embeddings
        outputs = self.backbone.deberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Use [CLS] token representation
        embeddings = outputs.last_hidden_state[:, 0, :]

        # Domain and fraud predictions
        domain_logits = self.domain_classifier(embeddings)
        fraud_logits = self.fraud_classifier(embeddings)

        # Calculate losses if labels provided
        loss = None
        if fraud_labels is not None and domain_labels is not None:
            fraud_loss = F.cross_entropy(fraud_logits, fraud_labels)
            domain_loss = F.cross_entropy(domain_logits, domain_labels)
            # Weighted combination (fraud detection is primary task)
            loss = 0.7 * fraud_loss + 0.3 * domain_loss

        result = {
            "loss": loss,
            "fraud_logits": fraud_logits,
            "domain_logits": domain_logits,
        }

        if return_embeddings:
            result["embeddings"] = embeddings

        return result


# TRAINING UTILITIES

In [13]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix

def compute_metrics(eval_pred):
    """Comprehensive metrics for fraud detection"""
    logits, labels = eval_pred

    # Handle multi-task case (fraud_logits, domain_logits)
    if isinstance(logits, (tuple, list)):
        fraud_logits = logits[0]
    else:
        fraud_logits = logits

    # Ensure fraud_logits is numpy array
    if isinstance(fraud_logits, torch.Tensor):
        fraud_logits = fraud_logits.detach().cpu().numpy()
    fraud_logits = np.atleast_2d(fraud_logits)

    # Ensure labels is numpy array
    labels = np.array(labels).flatten()

    # Predictions
    preds = np.argmax(fraud_logits, axis=1)
    probs = torch.softmax(torch.tensor(fraud_logits), dim=1)[:, 1].numpy()

    # Core metrics
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )

    # AUC-ROC
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.5

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "eval_accuracy": (preds == labels).mean(),
        "eval_f1": f1,
        "eval_precision": precision,
        "eval_recall": recall,
        "eval_auc": auc,
        "eval_specificity": specificity,
    }


In [14]:
class MultiTaskTrainer(Trainer):
    """Custom trainer for multi-task learning with class weights"""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        fraud_labels = inputs.pop("fraud_labels", None)
        domain_labels = inputs.pop("domain_labels", None)

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            fraud_labels=fraud_labels,
            domain_labels=domain_labels
        )

        # Apply class weights if provided
        if self.class_weights is not None and fraud_labels is not None:
            fraud_logits = outputs["fraud_logits"]
            weights = self.class_weights.to(fraud_logits.device)
            fraud_loss = F.cross_entropy(fraud_logits, fraud_labels, weight=weights)

            domain_logits = outputs["domain_logits"]
            domain_loss = F.cross_entropy(domain_logits, domain_labels)

            loss = 0.7 * fraud_loss + 0.3 * domain_loss
        else:
            loss = outputs["loss"]

        return (loss, outputs) if return_outputs else loss


        def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
            has_labels = all(inputs.get(k) is not None for k in ["fraud_labels", "domain_labels"])
            inputs = self._prepare_inputs(inputs)
            with torch.no_grad():
                if has_labels:
                    fraud_labels = inputs.pop("fraud_labels", None)
                    domain_labels = inputs.pop("domain_labels", None)
                    outputs = model(
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        fraud_labels=fraud_labels,
                        domain_labels=domain_labels
                    )
                    loss = outputs["loss"].detach()
                else:
                    loss = None
                    outputs = model(**inputs)
            fraud_logits = outputs["fraud_logits"].detach()
            labels = fraud_labels  # Use the pre-extracted fraud_labels
            if labels is not None:
                labels = labels.detach()
            return (loss, fraud_logits, labels)


# STAGE 1: SHARED ADAPTER TRAINING

In [15]:
class SharedAdapterTrainer:
    """Trains shared LoRA adapter on unified multi-domain dataset (batch-wise to save RAM)"""

    def __init__(self, config: Config, data_manager: DataManager):
        self.config = config
        self.data_manager = data_manager
        self.tokenizer = AutoTokenizer.from_pretrained(config.backbone_model)
        self.model = None
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    # ----------------------
    # Stage 0: Prepare lazy datasets
    # ----------------------
    def prepare_data_lazy(self, val_ratio=0.1):
        """
        Create lazy datasets that load batches from disk on-the-fly.
        val_ratio: fraction of data used for validation.
        """
        print("\n📊 Preparing lazy datasets...")

        domain_files = [Path(self.config.data_dir)/f"processed_{d}.pkl" for d in self.config.domains]

        # Lazy train dataset
        self.train_dataset = LazyMultiTaskDataset(
            domain_files, self.tokenizer, self.config.max_length, val_ratio=val_ratio, is_val=False
        )

        # Lazy validation dataset
        self.val_dataset = LazyMultiTaskDataset(
            domain_files, self.tokenizer, self.config.max_length, val_ratio=val_ratio, is_val=True
        )

        print(f"✅ Lazy train dataset: {len(self.train_dataset)} samples")
        print(f"✅ Lazy val dataset: {len(self.val_dataset)} samples")

    # ----------------------
    # Stage 1: Initialize model
    # ----------------------
    def initialize_model(self):
        print(f"\n🔧 Loading backbone: {self.config.backbone_model}")
        backbone = AutoModelForSequenceClassification.from_pretrained(
            self.config.backbone_model, num_labels=2
        )

        # Freeze backbone
        for param in backbone.parameters():
            param.requires_grad = False

        print("\n🎯 Applying shared LoRA adapter...")
        lora_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            target_modules=["query_proj", "value_proj", "key_proj"],
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type=TaskType.SEQ_CLS,
            inference_mode=False
        )
        backbone_with_lora = get_peft_model(backbone, lora_config)

        self.model = MultiTaskFraudDetector(backbone_with_lora, len(self.config.domains))
        self.model.to(device)

        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Trainable: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)")

    # ----------------------
    # Stage 2: Train model batch-wise
    # ----------------------
    def train_model(self, train_dataset=None, val_dataset=None):
        if train_dataset is None:
            train_dataset = self.train_dataset
        if val_dataset is None:
            val_dataset = self.val_dataset

        print("\n🚀 Starting training on lazy dataset...")
        training_args = TrainingArguments(
              output_dir=f"{self.config.output_dir}/shared_adapter",
              per_device_train_batch_size=self.config.batch_size,
              per_device_eval_batch_size=self.config.eval_batch_size,
              gradient_accumulation_steps=self.config.gradient_accumulation_steps,
              learning_rate=self.config.learning_rate,
              warmup_ratio=self.config.warmup_ratio,
              lr_scheduler_type="cosine",
              weight_decay=self.config.weight_decay,
              max_grad_norm=1.0,
              num_train_epochs=self.config.num_epochs,
              eval_strategy="steps",
              eval_steps=100,
              save_strategy="steps",
              save_steps=200,
              save_total_limit=2,
              load_best_model_at_end=True,
              metric_for_best_model="eval_f1",  # <-- match compute_metrics
              greater_is_better=True,
              fp16=True,
              dataloader_num_workers=0,
              dataloader_pin_memory=True,
              logging_steps=50,
              logging_strategy="steps",
              report_to="none",
              seed=42,
          )

        # Approximate class weights (can be refined for batch-wise computation)
        class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32)

        trainer = MultiTaskTrainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,  # ✅ pass validation dataset
            compute_metrics=compute_metrics,  # ✅ ensure compute_metrics is passed
            class_weights=class_weights
        )

        trainer.train()

        # Save shared adapter
        save_path = Path(self.config.adapter_dir) / "shared_adapter"
        self.model.backbone.save_pretrained(save_path)
        torch.save({
            "domain_classifier": self.model.domain_classifier.state_dict(),
            "fraud_classifier": self.model.fraud_classifier.state_dict(),
            "num_domains": len(self.config.domains),
            "domain_mapping": {i: d for i, d in enumerate(self.config.domains)}
        }, save_path / "task_heads.pt")
        print(f"\n💾 Shared adapter saved to: {save_path}")

        # Cleanup
        del trainer
        torch.cuda.empty_cache()


        # Save shared adapter
        save_path = Path(self.config.adapter_dir) / "shared_adapter"
        self.model.backbone.save_pretrained(save_path)
        torch.save({
            "domain_classifier": self.model.domain_classifier.state_dict(),
            "fraud_classifier": self.model.fraud_classifier.state_dict(),
            "num_domains": len(self.config.domains),
            "domain_mapping": {i: d for i, d in enumerate(self.config.domains)}
        }, save_path / "task_heads.pt")
        print(f"\n💾 Shared adapter saved to: {save_path}")

        # Cleanup
        del trainer
        torch.cuda.empty_cache()


# STAGE 2: DOMAIN-SPECIFIC ADAPTER TRAINING


In [16]:
class DomainSpecificTrainer:
    """Trains domain-specific adapters on top of shared adapter"""

    def __init__(self, config: Config, data_manager: DataManager):
        self.config = config
        self.data_manager = data_manager
        self.tokenizer = AutoTokenizer.from_pretrained(config.backbone_model)

    def train_domain(self, domain_name: str, shared_adapter_path: Path):
        print(f"\n{'='*70}")
        print(f"🎯 STAGE 2: TRAINING DOMAIN-SPECIFIC ADAPTER - {domain_name.upper()}")
        print(f"{'='*70}")

        # Load domain data
        df = self.data_manager.domain_data[domain_name]
        print(f"\n📊 Dataset: {len(df)} samples")

        # Class distribution
        class_counts = df["label"].value_counts().sort_index()
        print(f"  Class 0: {class_counts[0]} ({100*class_counts[0]/len(df):.1f}%)")
        print(f"  Class 1: {class_counts[1]} ({100*class_counts[1]/len(df):.1f}%)")

        # Class weights
        total = len(df)
        class_weights = torch.tensor([
            total / (2 * class_counts[0]),
            total / (2 * class_counts[1])
        ], dtype=torch.float32)

        # Split data
        train_df, temp_df = train_test_split(
            df, test_size=0.2, random_state=42, stratify=df["label"]
        )
        val_df, test_df = train_test_split(
            temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
        )

        # Create datasets
        train_dataset = FraudDataset(train_df)
        val_dataset = FraudDataset(val_df)
        test_dataset = FraudDataset(test_df)

        # Load backbone with shared adapter
        print(f"\n🔧 Loading backbone with shared adapter...")
        backbone = AutoModelForSequenceClassification.from_pretrained(
            self.config.backbone_model,
            num_labels=2
        )

        # Load shared adapter
        model = PeftModel.from_pretrained(
            backbone,
            shared_adapter_path,
            is_trainable=False  # Freeze shared adapter
        )

        # Add domain-specific adapter
        print(f"🎯 Adding domain-specific adapter for {domain_name}...")
        lora_config = LoraConfig(
            r=self.config.lora_r,
            lora_alpha=self.config.lora_alpha,
            target_modules=["query_proj", "value_proj", "key_proj"],
            lora_dropout=self.config.lora_dropout,
            bias="none",
            task_type=TaskType.SEQ_CLS,
            inference_mode=False
        )

        model.add_adapter(f"{domain_name}_adapter", lora_config)
        model.set_adapter(f"{domain_name}_adapter")
        model.to(device)

        # Training arguments
        training_args = TrainingArguments(
            output_dir=f"{self.config.output_dir}/{domain_name}",
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.eval_batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            learning_rate=self.config.learning_rate * 0.8,  # Slightly lower for fine-tuning
            warmup_ratio=0.1,
            lr_scheduler_type="cosine",
            weight_decay=self.config.weight_decay,
            num_train_epochs=3,  # Fewer epochs for domain-specific
            eval_strategy="steps",
            eval_steps=50,
            save_strategy="steps",
            save_steps=100,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            fp16=True,
            dataloader_num_workers=2,
            logging_steps=25,
            report_to="none",
            seed=42,
        )

        # Weighted trainer
        class WeightedLossTrainer(Trainer):
            def __init__(self, *args, class_weights=None, **kwargs):
                super().__init__(*args, **kwargs)
                self.class_weights = class_weights

            def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
                labels = inputs.pop("labels")
                outputs = model(**inputs)
                logits = outputs.logits

                if self.class_weights is not None:
                    weights = self.class_weights.to(logits.device)
                    loss = F.cross_entropy(logits, labels, weight=weights)
                else:
                    loss = outputs.loss

                return (loss, outputs) if return_outputs else loss

        trainer = WeightedLossTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            class_weights=class_weights,
            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=3,
                    early_stopping_threshold=0.001
                )
            ]
        )

        # Train
        print("\n🚀 Training...")
        trainer.train()

        # Evaluate
        print("\n📊 Test metrics:")
        test_metrics = trainer.evaluate(test_dataset)
        for key, value in test_metrics.items():
            if key.startswith("eval_"):
                print(f"   {key[5:]}: {value:.4f}")

        # Classification report
        print("\n📋 Classification Report:")
        test_preds = trainer.predict(test_dataset)
        y_pred = np.argmax(test_preds.predictions, axis=1)
        y_true = test_preds.label_ids
        print(classification_report(
            y_true, y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4
        ))

        # Save domain-specific adapter
        adapter_path = Path(self.config.adapter_dir) / f"{domain_name}_adapter"
        model.save_pretrained(adapter_path)
        print(f"\n💾 Domain adapter saved to: {adapter_path}")

        # Save results
        results = {
            "domain": domain_name,
            "test_metrics": test_metrics,
            "class_weights": class_weights.tolist()
        }

        with open(adapter_path / "results.json", "w") as f:
            json.dump(results, f, indent=2)

        # Cleanup
        del model, trainer
        torch.cuda.empty_cache()

        return results

    def train_all_domains(self):
        """Train all domain-specific adapters"""
        shared_adapter_path = Path(self.config.adapter_dir) / "shared_adapter"

        if not shared_adapter_path.exists():
            raise ValueError("Shared adapter not found! Train it first.")

        all_results = {}
        for domain in self.config.domains:
            try:
                results = self.train_domain(domain, shared_adapter_path)
                all_results[domain] = results
            except Exception as e:
                print(f"\n❌ ERROR training {domain}: {e}")
                import traceback
                traceback.print_exc()
                continue

        # Summary
        print("\n" + "="*70)
        print("📊 TRAINING COMPLETE - SUMMARY")
        print("="*70)

        summary_data = []
        for domain, results in all_results.items():
            summary_data.append({
                "Domain": domain,
                "Test F1": results["test_metrics"].get("eval_f1", 0),
                "Test Precision": results["test_metrics"].get("eval_precision", 0),
                "Test Recall": results["test_metrics"].get("eval_recall", 0),
                "Test AUC": results["test_metrics"].get("eval_auc", 0),
            })

        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
        summary_df.to_csv(f"{self.config.output_dir}/training_summary.csv", index=False)

        return all_results


# ENHANCED INFERENCE SYSTEM

In [17]:
class EnhancedFraudDetector:
    """
    Production-ready fraud detector with:
    - Automatic domain routing
    - Domain-specific adapters
    - Ensemble predictions
    """

    def __init__(self, config: Config):
        self.config = config
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(config.backbone_model)

        # Load shared adapter with domain classifier
        self._load_shared_model()

        # Cache for domain-specific models
        self.domain_models = {}

    def _load_shared_model(self):
        """Load shared adapter with domain classifier"""
        print("\n🔧 Loading shared model with domain classifier...")

        # Load backbone
        backbone = AutoModelForSequenceClassification.from_pretrained(
            self.config.backbone_model,
            num_labels=2
        )

        # Load shared adapter
        shared_path = Path(self.config.adapter_dir) / "shared_adapter"
        backbone_with_lora = PeftModel.from_pretrained(backbone, shared_path)

        # Load task heads
        heads_path = shared_path / "task_heads.pt"
        heads_data = torch.load(heads_path, map_location=self.device)

        # Reconstruct multi-task model
        self.shared_model = MultiTaskFraudDetector(
            backbone_with_lora,
            heads_data["num_domains"]
        )
        self.shared_model.domain_classifier.load_state_dict(heads_data["domain_classifier"])
        self.shared_model.fraud_classifier.load_state_dict(heads_data["fraud_classifier"])
        self.shared_model.to(self.device)
        self.shared_model.eval()

        self.domain_mapping = heads_data["domain_mapping"]
        self.reverse_domain_mapping = {v: k for k, v in self.domain_mapping.items()}

        print("✅ Shared model loaded")

    def _load_domain_model(self, domain_name: str):
        """Load domain-specific adapter"""
        if domain_name in self.domain_models:
            return self.domain_models[domain_name]

        print(f"🔧 Loading domain adapter: {domain_name}")

        # Load backbone with shared adapter
        backbone = AutoModelForSequenceClassification.from_pretrained(
            self.config.backbone_model,
            num_labels=2
        )
        shared_path = Path(self.config.adapter_dir) / "shared_adapter"
        model = PeftModel.from_pretrained(backbone, shared_path)

        # Load domain-specific adapter
        domain_path = Path(self.config.adapter_dir) / f"{domain_name}_adapter"
        if domain_path.exists():
            model = PeftModel.from_pretrained(
                model.base_model,
                domain_path,
                adapter_name=f"{domain_name}_adapter"
            )
            model.set_adapter(f"{domain_name}_adapter")

        model.to(self.device)
        model.eval()

        self.domain_models[domain_name] = model
        return model

    def detect_domain(self, text: str, return_probs: bool = False) -> Dict:
        """Detect which domain the text belongs to"""
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_length,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.shared_model(**inputs)
            domain_logits = outputs["domain_logits"]
            domain_probs = F.softmax(domain_logits, dim=-1)[0]

            predicted_domain_id = domain_probs.argmax().item()
            predicted_domain = self.domain_mapping[predicted_domain_id]
            confidence = domain_probs[predicted_domain_id].item()

        result = {
            "predicted_domain": predicted_domain,
            "confidence": confidence
        }

        if return_probs:
            result["all_probabilities"] = {
                self.domain_mapping[i]: prob.item()
                for i, prob in enumerate(domain_probs)
            }

        return result

    def predict_fraud(
        self,
        text: str,
        domain: Optional[str] = None,
        use_domain_adapter: bool = True,
        return_details: bool = False
    ) -> Dict:
        """
        Predict fraud probability

        Args:
            text: Input text
            domain: Specific domain (if None, auto-detect)
            use_domain_adapter: Use domain-specific adapter
            return_details: Return detailed predictions
        """
        # Detect domain if not provided
        if domain is None:
            domain_result = self.detect_domain(text, return_probs=True)
            domain = domain_result["predicted_domain"]
            domain_confidence = domain_result["confidence"]
        else:
            domain_confidence = 1.0

        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=self.config.max_length,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Choose model
        if use_domain_adapter:
            model = self._load_domain_model(domain)
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits
        else:
            with torch.no_grad():
                outputs = self.shared_model(**inputs)
                logits = outputs["fraud_logits"]

        # Get predictions
        probs = F.softmax(logits, dim=-1)[0]
        fraud_prob = probs[1].item()
        legit_prob = probs[0].item()
        prediction = 1 if fraud_prob > 0.5 else 0

        result = {
            "text": text[:100] + "..." if len(text) > 100 else text,
            "domain": domain,
            "prediction": "FRAUD" if prediction == 1 else "LEGITIMATE",
            "fraud_probability": fraud_prob,
            "confidence": max(fraud_prob, legit_prob),
        }

        if return_details:
            result["legitimate_probability"] = legit_prob
            result["domain_confidence"] = domain_confidence
            result["model_used"] = "domain_specific" if use_domain_adapter else "shared"

        return result

    def ensemble_predict(
        self,
        text: str,
        domains: Optional[List[str]] = None,
        aggregation: str = "weighted_avg"
    ) -> Dict:
        """
        Ensemble prediction across multiple domains

        Args:
            text: Input text
            domains: List of domains to use (if None, use top-3 by domain classifier)
            aggregation: 'weighted_avg', 'max', or 'voting'
        """
        # Auto-select domains if not provided
        if domains is None:
            domain_result = self.detect_domain(text, return_probs=True)
            all_probs = domain_result["all_probabilities"]
            # Use top 3 domains by confidence
            domains = sorted(
                all_probs.items(),
                key=lambda x: x[1],
                reverse=True
            )[:3]
            domains = [d[0] for d in domains]
            domain_weights = [all_probs[d] for d in domains]
        else:
            domain_weights = [1.0] * len(domains)

        # Normalize weights
        total_weight = sum(domain_weights)
        domain_weights = [w / total_weight for w in domain_weights]

        # Get predictions from each domain
        predictions = []
        for domain, weight in zip(domains, domain_weights):
            try:
                result = self.predict_fraud(
                    text,
                    domain=domain,
                    use_domain_adapter=True,
                    return_details=True
                )
                predictions.append({
                    "domain": domain,
                    "fraud_prob": result["fraud_probability"],
                    "weight": weight
                })
            except Exception as e:
                print(f"⚠️ Error predicting with {domain}: {e}")
                continue

        if not predictions:
            raise ValueError("No valid predictions obtained")

        # Aggregate
        if aggregation == "weighted_avg":
            final_fraud_prob = sum(
                p["fraud_prob"] * p["weight"] for p in predictions
            )
        elif aggregation == "max":
            final_fraud_prob = max(p["fraud_prob"] for p in predictions)
        elif aggregation == "voting":
            votes = sum(1 for p in predictions if p["fraud_prob"] > 0.5)
            final_fraud_prob = votes / len(predictions)
        else:
            raise ValueError(f"Unknown aggregation: {aggregation}")

        return {
            "text": text[:100] + "..." if len(text) > 100 else text,
            "ensemble_prediction": "FRAUD" if final_fraud_prob > 0.5 else "LEGITIMATE",
            "fraud_probability": final_fraud_prob,
            "confidence": max(final_fraud_prob, 1 - final_fraud_prob),
            "individual_predictions": predictions,
            "aggregation_method": aggregation
        }



# MAIN EXECUTION PIPELINE

In [ ]:
def main():
    """Main training pipeline"""

    # Initialize configuration
    config = Config()

    # Load data
    print("="*70)
    print("🚀 ENHANCED MULTI-DOMAIN FRAUD DETECTION SYSTEM")
    print("="*70)

    data_manager = DataManager(config)
    data_manager.load_all_domains()

    # Show statistics
    print("\n📊 Dataset Statistics:")
    stats = data_manager.get_domain_stats()
    print(stats.to_string(index=False))

    # STAGE 1: Train shared adapter
    shared_trainer = SharedAdapterTrainer(config, data_manager)
    shared_model = shared_trainer.train()

    del shared_model
    torch.cuda.empty_cache()

    # STAGE 2: Train domain-specific adapters
    domain_trainer = DomainSpecificTrainer(config, data_manager)
    results = domain_trainer.train_all_domains()

    print("\n✅ Training pipeline complete!")
    return results

# USAGE EXAMPLES

In [ ]:
def demo_inference():
    """Demonstrate inference capabilities"""

    print("\n" + "="*70)
    print("🎯 INFERENCE DEMO")
    print("="*70)

    # Initialize detector
    config = Config()
    detector = EnhancedFraudDetector(config)

    # Test cases
    test_cases = [
        {
            "text": "URGENT! Your account will be suspended. Click here NOW to verify!",
            "expected_domain": "phishing"
        },
        {
            "text": "Breaking: Scientists discover aliens on Mars! Government hiding truth!",
            "expected_domain": "fake_news"
        },
        {
            "text": "Work from home! Earn $5000/week with no experience required!",
            "expected_domain": "job_scams"
        },
        {
            "text": "This product changed my life! 5 stars! Amazing quality and fast shipping.",
            "expected_domain": "product_reviews"
        }
    ]

    print("\n🔍 Test Case 1: Auto-detect domain")
    result = detector.predict_fraud(
        test_cases[0]["text"],
        return_details=True
    )
    print(f"Text: {result['text']}")
    print(f"Detected Domain: {result['domain']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Fraud Probability: {result['fraud_probability']:.2%}")
    print(f"Confidence: {result['confidence']:.2%}")

    print("\n🔍 Test Case 2: Specify domain")
    result = detector.predict_fraud(
        test_cases[1]["text"],
        domain="fake_news",
        return_details=True
    )
    print(f"Text: {result['text']}")
    print(f"Domain: {result['domain']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Fraud Probability: {result['fraud_probability']:.2%}")

    print("\n🔍 Test Case 3: Ensemble prediction")
    result = detector.ensemble_predict(
        test_cases[2]["text"],
        aggregation="weighted_avg"
    )
    print(f"Text: {result['text']}")
    print(f"Ensemble Prediction: {result['ensemble_prediction']}")
    print(f"Fraud Probability: {result['fraud_probability']:.2%}")
    print(f"Domains used: {[p['domain'] for p in result['individual_predictions']]}")

    print("\n🔍 Test Case 4: Domain detection only")
    result = detector.detect_domain(test_cases[3]["text"], return_probs=True)
    print(f"Predicted Domain: {result['predicted_domain']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print("Top 3 domains:")
    sorted_probs = sorted(
        result['all_probabilities'].items(),
        key=lambda x: x[1],
        reverse=True
    )[:3]
    for domain, prob in sorted_probs:
        print(f"  {domain}: {prob:.2%}")



# RUN

In [ ]:
if __name__ == "__main__":
    # Uncomment to run full training pipeline
    # results = main()

    # Uncomment to run inference demo (after training)
    # demo_inference()

    print("\n📝 To run:")
    print("1. Training: results = main()")
    print("2. Inference: demo_inference()")

In [18]:
# ==============================
# 🔧 Initialize Configuration
# ==============================
config = Config()
print("✅ Configuration initialized")


✅ Configuration initialized


In [21]:
# ==============================
# 📂 Load Domain Data
# ==============================
data_manager = DataManager(config)
all_data = data_manager.load_all_domains()



📂 Loading domain datasets...
  ✅ phishing: 15272 samples
  ✅ fake_news: 20456 samples
  ✅ political_statements: 12497 samples
  ✅ product_reviews: 20971 samples
  ✅ job_scams: 14295 samples
  ✅ sms: 6574 samples
  ✅ twitter_rumours: 5789 samples

✅ Loaded 7 domains successfully


In [22]:
# ==============================
# 📊 Dataset Statistics
# ==============================
stats = data_manager.get_domain_stats()
print(stats.to_string(index=False))


              domain  total  legitimate  fraud  fraud_ratio
            phishing  15272        9198   6074     0.397721
           fake_news  20456       11624   8832     0.431756
political_statements  12497        4455   8042     0.643514
     product_reviews  20971       10479  10492     0.500310
           job_scams  14295       13696    599     0.041903
                 sms   6574        5300   1274     0.193794
     twitter_rumours   5789        3820   1969     0.340128


In [23]:
# ==============================
# 🏗️ Stage 1: Train Shared Adapter
# ==============================
shared_trainer = SharedAdapterTrainer(config, data_manager)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [24]:
shared_trainer.prepare_data_lazy(val_ratio=0.1)


📊 Preparing lazy datasets...
✅ Lazy train dataset: 86272 samples
✅ Lazy val dataset: 9582 samples


In [25]:
shared_trainer.initialize_model()


🔧 Loading backbone: microsoft/deberta-v3-small


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🎯 Applying shared LoRA adapter...
  Trainable: 816,779 / 142,713,229 (0.57%)


In [26]:
shared_trainer.train_model()


🚀 Starting training on lazy dataset...


Step,Training Loss,Validation Loss


ValueError: Found input variables with inconsistent numbers of samples: [19164, 9582]

In [ ]:
del shared_trainer
torch.cuda.empty_cache()

In [ ]:
# ==============================
# 🏗️ Stage 2: Train Domain-Specific Adapters
# ==============================
domain_trainer = DomainSpecificTrainer(config, data_manager)
results = domain_trainer.train_all_domains()
print("\n✅ Training pipeline complete!")


In [ ]:
import pickle
with open(config.output_dir + "/training_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("✅ Results saved to outputs folder")
